1-vazifa: Ma’lumotlarni tayyorlash

maktab so‘zi uchun ma’lumotlarni tayyorlang.





"maktab" so‘zini sequence o‘zgaruvchisiga yuklang.



So‘zdan unikal harflar lug‘atini (chars) yarating.<br>



Harflarni indeksga (char2idx) va indekslarni harfga (idx2char) o‘giruvchi lug‘atlarni yarating.



Kiruvchi ma’lumot (x_data) va kutilayotgan natija (y_data) uchun ro‘yxatlarni yarating. Eslab qoling, x_data oxirgi harfsiz, y_data esa birinchi harfsiz bo‘lishi kerak.



Tayyorlangan x_data va y_data ni tensorlarga aylantiring va unsqueeze() yordamida shaklini o‘zgartiring.



2-vazifa: Modelni yaratish

Videoda ko‘rsatilgan HarfRNN klassini yarating. Model arxitekturasi (RNN va Chiziqli qatlamlar) o‘zgarishsiz qolishi kerak. Faqat yangi vocab_size ga moslashishini ta’minlang.





nn.Module dan meros oluvchi HarfRNN klassini yarating.



__init__ metodida nn.RNN va nn.Linear qatlamlarini e’lon qiling.



forward metodini yozing, u x va hidden holatlarini qabul qilib, natija (out) va yangi hidden holatini qaytarsin.



3-vazifa: Modelni o‘qitish

Modelni yangi giperparametrlar bilan o‘qiting.<br><br>1. hidden_size ni 10 ga, lr (o‘rganish tezligi) ni 0.02 ga va epochs (epoxalar) sonini 200 ga o‘rnating.





hidden_size ni 10 ga, lr (o‘rganish tezligi) ni 0.02 ga va epochs (epoxalar) sonini 200 ga o‘rnating.



Model, yo‘qotish funksiyasi (criterion), va optimizatorni (optimizer) e’lon qiling.



O‘qitish siklini yarating. Sikl ichida har bir element uchun yo‘qotishni hisoblang, gradientlarni nollang, loss.backward() va optimizer.step() ni chaqiring.



Har 20 epoxada yo‘qotish qiymati va model bashorat qilgan ketma-ketlikni (pred_seq) konsolga chiqaring.



4-vazifa: Natijalarni tahlil qilish

3-vazifada olingan natijalarga qarab quyidagi savolga javob bering: <br><br>"Model maktab so‘zining keyingi harflarini to‘g‘ri bashorat qilishni o‘rgandimi? Javobingizni epoxalar davomida yo‘qotish (Loss) qiymatining o‘zgarishi va oxirgi bashorat (Pred) natijasiga tayanib asoslang."

In [8]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
#torch — tensorlar bilan ishlash uchun
#torch.nn — neyron tarmoq qatlamlari uchun
#torch.optim — optimizatorlar uchun
#torch.nn.functional — one_hot kabi yordamchi funksiyalar uchun

In [9]:
# So'z
sequence = "maktab"

chars = sorted(list(set(sequence)))
print("Unikal harflar:", chars)
  #set(sequence) → takroriy harflarni olib tashlaydi
  #list(...) → uni ro‘yxatga aylantiradi
  #sorted(...) → tartiblaydi

# Harf -> indeks
char2idx = {ch: i for i, ch in enumerate(chars)}

# Indeks -> harf
idx2char = {i: ch for ch, i in char2idx.items()}

print("char2idx:", char2idx)
print("idx2char:", idx2char)

# x_data = oxirgi harfsiz
# y_data = birinchi harfsiz
x_data = [char2idx[ch] for ch in sequence[:-1]]
y_data = [char2idx[ch] for ch in sequence[1:]]

print("x_data:", x_data)
print("y_data:", y_data)

# Tensorlarga aylantirish
x_tensor = torch.tensor(x_data, dtype=torch.long).unsqueeze(1)
y_tensor = torch.tensor(y_data, dtype=torch.long).unsqueeze(1)

print("x_tensor shape:", x_tensor.shape)
print("y_tensor shape:", y_tensor.shape)

vocab_size = len(chars)
print("vocab_size:", vocab_size)

Unikal harflar: ['a', 'b', 'k', 'm', 't']
char2idx: {'a': 0, 'b': 1, 'k': 2, 'm': 3, 't': 4}
idx2char: {0: 'a', 1: 'b', 2: 'k', 3: 'm', 4: 't'}
x_data: [3, 0, 2, 4, 0]
y_data: [0, 2, 4, 0, 1]
x_tensor shape: torch.Size([5, 1])
y_tensor shape: torch.Size([5, 1])
vocab_size: 5


In [10]:
# Modelni yaratish
class HarfRNN(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super(HarfRNN, self).__init__()
        self.hidden_size = hidden_size
        #input_size — kiruvchi vektor hajmi
        #hidden_size — yashirin qatlam hajmi
        #output_size — chiqish hajmi

        self.rnn = nn.RNN(input_size, hidden_size)
            #kirish o‘lchami = input_size
            #yashirin holat o‘lchami = hidden_size
        self.fc = nn.Linear(hidden_size, output_size)
            #RNN chiqargan yashirin holatni oxirgi natijaga aylantiradi.

    def forward(self, x, hidden):
        out, hidden = self.rnn(x, hidden)
        out = self.fc(out)
        return out, hidden

In [11]:
# Giperparametrlar va model
hidden_size = 10
lr = 0.02
epochs = 200

model = HarfRNN(input_size=vocab_size, hidden_size=hidden_size, output_size=vocab_size)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=lr)

Modelni o'qitish

In [12]:
for epoch in range(1, epochs + 1):
    hidden = torch.zeros(1, 1, hidden_size)
    loss = 0

    optimizer.zero_grad()
    outputs = []

    for i in range(len(x_tensor)):
        # One-hot encoding
        x_onehot = F.one_hot(x_tensor[i], num_classes=vocab_size).float()
        x_onehot = x_onehot.unsqueeze(0)  # shape: [1, 1, vocab_size]

        out, hidden = model(x_onehot, hidden)
        outputs.append(out)

        loss += criterion(out.squeeze(0), y_tensor[i])

    loss.backward()
    optimizer.step()

    if epoch % 20 == 0:
        pred_seq = ""
        for out in outputs:
            pred_idx = torch.argmax(out.squeeze(0), dim=1).item()
            pred_seq += idx2char[pred_idx]

        print(f"Epoch [{epoch}/{epochs}], Loss: {loss.item():.4f}, Pred: {pred_seq}")

Epoch [20/200], Loss: 0.8342, Pred: aktab
Epoch [40/200], Loss: 0.0553, Pred: aktab
Epoch [60/200], Loss: 0.0235, Pred: aktab
Epoch [80/200], Loss: 0.0168, Pred: aktab
Epoch [100/200], Loss: 0.0135, Pred: aktab
Epoch [120/200], Loss: 0.0113, Pred: aktab
Epoch [140/200], Loss: 0.0096, Pred: aktab
Epoch [160/200], Loss: 0.0082, Pred: aktab
Epoch [180/200], Loss: 0.0071, Pred: aktab
Epoch [200/200], Loss: 0.0063, Pred: aktab


Oxirgi bashorat

In [13]:
hidden = torch.zeros(1, 1, hidden_size)
pred_seq = ""

for i in range(len(x_tensor)):
    x_onehot = F.one_hot(x_tensor[i], num_classes=vocab_size).float()
    x_onehot = x_onehot.unsqueeze(0)

    out, hidden = model(x_onehot, hidden)
    pred_idx = torch.argmax(out.squeeze(0), dim=1).item()
    pred_seq += idx2char[pred_idx]

print("Oxirgi bashorat:", pred_seq)
print("To'g'ri javob:", sequence[1:])

Oxirgi bashorat: aktab
To'g'ri javob: aktab


Model `maktab` so‘zining keyingi harflarini bashorat qilishni o‘rgandi. Buni `Loss` qiymatining epoxalar davomida kamayib borishidan bilish mumkin. Oxirgi epoxalarda modelning `Pred` natijasi `aktab` ga yaqinlashadi yoki aynan shu qiymatni beradi. Bu esa model harflar ketma-ketligi orasidagi bog‘lanishni muvaffaqiyatli o‘rganganini ko‘rsatadi.